# 🇹🇭 Thai NLP Toolkit — Training on Colab (Simplified Version)

Notebook นี้ออกแบบมาให้ใช้งานง่ายที่สุดและป้องกันปัญหาการหลุดการเชื่อมต่อ (Disconnect):
1. Clone โปรเจคจาก GitHub
2. ติดตั้ง library
3. ดาวน์โหลด datasets ล่าสุดจาก Hugging Face (อัตโนมัติ)
4. อัปโหลดเฉพาะโมเดล Tokenizer
5. เชื่อมต่อ Google Drive เพื่อเซฟโมเดลโดยตรง (ป้องกันข้อมูลสูญหายเมื่อ Colab หลุด)
6. เริ่มต้นเทรนด้วย GPU

> ⚠️ **ก่อนเริ่ม**: ไปที่เมนูด้านบน **Runtime (รันไทม์) → Change runtime type (เปลี่ยนประเภทการรันไทม์) → GPU (T4)**

## 1. ตรวจสอบ GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Clone โปรเจคจาก GitHub

In [ ]:
# ทำการย้ายไปที่โฟลเดอร์หลัก /content เสมอเพื่อป้องกันการ Clone ซ้อนโฟลเดอร์เดิม
%cd /content
import os, shutil
if os.path.exists("thai-nlp-toolkit"):
    shutil.rmtree("thai-nlp-toolkit")
!git clone https://github.com/puttibenz/thai-nlp-toolkit.git
%cd thai-nlp-toolkit

## 3. ติดตั้ง Dependencies

In [ ]:
!pip install -q sentencepiece>=0.1.99 pythainlp>=4.0.0 datasets>=2.14.0 pyyaml>=6.0 tqdm>=4.65.0 scikit-learn>=1.3.0

## 4. ดาวน์โหลด Raw Datasets (อัตโนมัติ)

In [ ]:
# ใช้ -m เพื่อไม่ให้ Python สนสับสน path กับโฟลเดอร์ data/datasets.py ของโปรเจค
!python -m data.download --task all

## 5. อัปโหลดโมเดล Tokenizer

รันเซลล์ด้านล่างนี้แล้วเลือกอัปโหลดไฟล์ **`thai_bpe.model`** และ **`thai_bpe.vocab`** (อยู่ในโฟลเดอร์ `tokenizer` บนเครื่องของคุณ):

In [ ]:
import os
from google.colab import files

print("กรุณาเลือกไฟล์ 'thai_bpe.model' และ 'thai_bpe.vocab' เพื่ออัปโหลด:")
uploaded = files.upload()

os.makedirs("tokenizer", exist_ok=True)
for filename in uploaded.keys():
    dest = os.path.join("tokenizer", filename)
    if os.path.exists(dest):
        os.remove(dest)
    os.rename(filename, dest)
    print(f"✅ ย้าย {filename} เข้าโฟลเดอร์ tokenizer/ สำเร็จ")

## 6. เชื่อมต่อ Google Drive (แนะนำอย่างยิ่ง ⭐)

แนะนำให้เชื่อมต่อ Google Drive เพื่อสั่งเซฟโมเดล checkpoints ลงไดรฟ์โดยตรง ป้องกันประวัติการเทรนหายหาก Colab หลุดการเชื่อมต่อระหว่างทาง

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# สร้างโฟลเดอร์เก็บ output บน Google Drive ของคุณ
!mkdir -p /content/drive/MyDrive/thai_nlp_outputs

## 7. ตรวจสอบความพร้อมของไฟล์

In [ ]:
import os

required_files = [
    "tokenizer/thai_bpe.model",
    "tokenizer/thai_bpe.vocab",
    "tokenizer/tokenizer_config.json",
    "data/raw/ner_train.jsonl",
    "data/raw/sent_train.tsv",
    "data/raw/qa_train.json",
]

for f in required_files:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"{status}  {f}")

## 8. เริ่มต้น Training 🚀

### ทางเลือก A: บันทึกโมเดลลง Google Drive โดยตรง (แนะนำที่สุด ⭐)
วิธีนี้จะเซฟไฟล์โมเดลไปที่ Google Drive ของคุณทุกๆ 1,000 steps หาก Colab หลุดการเชื่อมต่อ คุณสามารถรันโค้ดและส่ง `--resume` โหลดเซฟเก่ามาทำต่อได้ทันที

In [ ]:
# เทรนและเซฟไว้ใน Google Drive
!python -m scripts.train --device cuda --output_dir /content/drive/MyDrive/thai_nlp_outputs

*(หากเทรนหลุดและต้องการรันต่อจากสเตปล่าสุด ให้ยกเลิก comment บรรทัดด้านล่างแล้วรัน)*

In [ ]:
# !python -m scripts.train --device cuda --output_dir /content/drive/MyDrive/thai_nlp_outputs --resume /content/drive/MyDrive/thai_nlp_outputs/checkpoint_best

### ทางเลือก B: บันทึกโมเดลไว้บน Colab เท่านั้น (หากไม่อยากต่อ Drive)
⚠️ *ระวัง: หากเน็ตหลุดหรือปิดเบราว์เซอร์ ข้อมูลจะหายทั้งหมด*

In [ ]:
# เทรนและเซฟไว้บนเครื่องจำลองของ Colab เท่านั้น
!python -m scripts.train --device cuda